# HW 4 
By Eugenia Tate for AIPI 530

## Problem 1: Campus help-desk staffing

Load packages 

In [1]:
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

Create Data

In [2]:
shiftData = pd.DataFrame(
    {"Shift": ["A", "B", "C", "D", "E"], "Cost": [120, 120, 120, 140, 210]}
)

demandData = pd.DataFrame({"Period": [1, 2, 3, 4], "Demand": [4, 6, 5, 3]})

coverageData = pd.DataFrame(
    {
        "A": [1, 1, 0, 0],
        "B": [0, 1, 1, 0],
        "C": [0, 0, 1, 1],
        "D": [1, 0, 0, 1],
        "E": [1, 1, 1, 1],
    },
    index=[1, 2, 3, 4],
)

coverageData.index.name = "Period"

paramData = pd.DataFrame({"maxE": [2]})

print(shiftData, demandData, coverageData, paramData, sep="\n\n")

  Shift  Cost
0     A   120
1     B   120
2     C   120
3     D   140
4     E   210

   Period  Demand
0       1       4
1       2       6
2       3       5
3       4       3

        A  B  C  D  E
Period               
1       1  0  0  1  1
2       1  1  0  0  1
3       0  1  1  0  1
4       0  0  1  1  1

   maxE
0     2


Define Model

In [3]:
m = AbstractModel()

# define sets
m.SHIFTS = Set()
m.PERIODS = Set()

# define parameters
m.cost = Param(m.SHIFTS)
m.demand = Param(m.PERIODS)
m.coverage = Param(m.PERIODS, m.SHIFTS)
m.maxE = Param()

# define decision variable
m.NShift = Var(m.SHIFTS, domain=NonNegativeIntegers)

# define implicit variable
m.NWorking = Var(m.PERIODS, domain=NonNegativeReals)

# define period staffing
def NWorking_Rule(m, t):
    return m.NWorking[t] == sum(m.coverage[t, s] * m.NShift[s] for s in m.SHIFTS)

m.Define_NWorking = Constraint(m.PERIODS, rule=NWorking_Rule)


# meet staffing requirements
def Demand_Rule(m, t):
    return m.NWorking[t] >= m.demand[t]

m.Cover_Demand = Constraint(m.PERIODS, rule=Demand_Rule)


# limit pattern availability
def Availability_Rule(m):
    return m.NShift["E"] <= m.maxE

m.Limit_E = Constraint(rule=Availability_Rule)


# minimize total payroll
def Min_Payroll_Rule(m):
    return sum(m.cost[s] * m.NShift[s] for s in m.SHIFTS)

m.Min_Payroll = Objective(rule=Min_Payroll_Rule, sense=minimize)

### Create instance

In [4]:
instanceData = {
    None: {
        "SHIFTS": {None: shiftData["Shift"].tolist()},
        "PERIODS": {None: demandData["Period"].tolist()},
        "cost": shiftData.set_index("Shift")["Cost"].to_dict(),
        "demand": demandData.set_index("Period")["Demand"].to_dict(),
        "coverage": coverageData.stack().to_dict(),
        "maxE": {None: paramData.loc[0, "maxE"]},
    }
}

instance = m.create_instance(instanceData)

Solve

In [5]:
solver = SolverFactory("glpk")
solution = solver.solve(instance)

assert check_optimal_termination(solution), "Solver did not terminate optimally."

Print Solution

In [6]:
print(solution["Solver"])

output = pd.DataFrame(
    [{"Shift": s, "Employees": value(instance.NShift[s])} for s in instance.SHIFTS]
)

coverage = pd.DataFrame(
    [
        {
            "Period": t,
            "Required": value(instance.demand[t]),
            "Working": value(instance.NWorking[t]),
            "Excess": (value(instance.NWorking[t]) - value(instance.demand[t])),
        }
        for t in instance.PERIODS
    ]
)

print(output.round(4).to_string(index=False))
print(coverage.round(4).to_string(index=False))
print(f"Total Payroll: ${value(instance.Min_Payroll):,.2f}")


- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 1
      Number of created subproblems: 1
  Error rc: 0
  Time: 0.03267407417297363

Shift  Employees
    A        2.0
    B        2.0
    C        1.0
    D        0.0
    E        2.0
 Period  Required  Working  Excess
      1         4      4.0     0.0
      2         6      6.0     0.0
      3         5      5.0     0.0
      4         3      3.0     0.0
Total Payroll: $1,020.00


**Explanation of results**

Assign 2 employees to A, 2 to B, 1 to C, none to D, and 2 to E.
The minimum total payroll is $1,020.

Coverage is 4, 6, 5, and 3 employees across the four periods, so
excess coverage is zero throughout. All four staffing requirements
and the two-employee availability limit for E are binding.

GLPK reports an optimal solution, certifying the global minimum.

## Problem 2: Nutrition Blend